# Py-CoSim
EnergyPlus Python EMS-Based Co-Simulation Framework\
Author: Liang Zhang @ UArizona\
The is a work in progress. Check all the TODOs\
This is cleaned version based on `history_versions_2/example_run_clean_version_add_ML_control-mannual input_and_chatgpt_disabled_DL_occ_change_htg_to_clg_update_dir_PyCoSim.ipynb`

## Tested Versions
EnergyPlus Version: 22.2.0\
EnergyPlus API version: 0.2\
Python: 3.8.2

## Installation
Download and install the latest version of [Conda](https://www.anaconda.com/products/distribution)\
Run Anaconda Prompt as Administrator\
Create a new conda environment:\
`$ conda create -n <enter-the-name-of-repository-here> python=3.8.2 pip`
`$ activate <enter-the-name-of-repository-here>`
Ensure that you have navigated to the top level of your cloned repository. You will execute all your pip commands from this location. For example:\
`$ cd /path/to/repository`
Install the environment needed for this repository:\
`$ pip install -e .[dev]`

### Potential Usage
For single buildings
1. Test latest ML and control algorithms for single bulidings

For multiple buildings (community)
1. Community-scale co-simulation and operations among buildings, DER, ESS, and interdisciplianry components

### Connection with Other Components
1. Distributed Energy Resources (DER)
2. Energy Storage (Battery and Ice Storage)
3. Decision Making and Optimization Component
4. Artificial Intellegence, such as deep learning models (DL) and large language models (LLM)

### Paper
1. `Box\Liang and Alyssa Collaboration Folder\Python-EMS-Based Building Simulation Infrastructure`

### Resources
1. https://github.com/jmarrec/OpenStudio_to_EnergyPlusAPI/blob/main/OpenStudio_to_EnergyPlusAPI.ipynb
2. Py EMS API documentation
https://nrel.github.io/EnergyPlus/api/python/datatransfer.html
3. Available variable handles: EnergyPlusV22-2-0 : Input Output Reference: Page 2831

### TODO
1. function_n_data (bulding components)
- TODO: Inside function_n_data, eppy is used to identify all zone names and controllables
- TODO: use decorator
- TODO: export facility electricity, but don't know what is the variable handels.
- TODO: need a standardized timestamp. Should the timestamp be a global variable?
- TODO: used to use number of files to control timestep of two EnergyPlus models, next step should be using timestamp to sync all EnergyPlus models
2. Message Systems
- Use existing Python package 1) ZeroMQ, 2) RabbitMQ, and 3) pypubsub
3. Component Prioritazation
- Add attribute of prioritation for each component to decide the order of component run
4. Multiprocessing
- With threading, communication between running multiple instances is quite easy as you can just communicate using a messaging library like pypubsub to initiate signaling between the threads. With multiprocessing, communication is more difficult, but there are likely ways around it.
- mpi4py: "I am currently also working on a parallel programming project using the E+-API in Python.We are using mpi4py for the communication between the different instances of the E+-Buildings. We wanted to divide on core into multiple threads, and on each thread we could simulate one building at the same time. But in python there is the Global Interpreter lock, which usually is not allowing any multithreading applications.(https://realpython.com/python-gil/)"
5. TODOs in the code
6. Others
- Rhinoceros, Grasshopper, honeybee??Honeybee connects Rhino geometry and Grasshopper functionality to a selection of advanced energy modeling and simulation programs such as Radiance, Daysim, EnergyPlus, OpenStudio, and gbXML.
- Work with dompc: https://github.com/do-mpc/do-mpc
- Work with NeuroMANCER: https://github.com/pnnl/neuromancer
- In terms of parallel computing: another way to do it is to create multiple python script and run them together through bash: e.g., "python script1.py & python script2.py"
- Modualize
- hardcode year to 2009
- Docker
- Stability of online visualization

In [1]:
# built-in envrionment
import glob
import time
import datetime
import sys
sys.path.insert(0, 'C:/EnergyPlusV22-2-0/') # add EnergyPlus directory to system path
import os
working_path = os.getcwd() # get working path


# Py EMS modules
from pyenergyplus.api import EnergyPlusAPI

# Enable online plotting in Jupyter Notebook
%matplotlib notebook
import matplotlib.pyplot as plt

# basic data science package
import numpy as np
import pandas as pd

# Make co-simulation without randomness
# np.random.seed(123)

# threading for run components at the same time
from threading import Thread

# Enable Interactive user input
import tkinter as tk
from tkinter import simpledialog

# OpenAI to use LLM
import openai

# Pytorch for deep learning
import torch.nn as nn
import torch
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE' #enable the threading to work with PyTorch

#packages about IML
np.random.seed(42)
import pickle

In [2]:
# generate random cooling setpoint
clg_random = np.random.uniform(22, 30, size=(1000,1))

### Building 1 Class

In [3]:
class function_n_data:
    
    """
    __init__: defines class inputs and variables
    init_plot: defines the initial status of online visualization
    update_line: defines how online visualization is updated
    callback_function: the key function that define the interaction between Python and EnergyPlus
    """

    def __init__(self, building_id, building_info, filename_to_run, weather_file, result_folder):
        # Class inputs
        ## Building Characteristics
        self.building_id = building_id
        self.building_info = building_info
        ## Define directories and files
        self.filename_to_run = filename_to_run
        self.weather_file = weather_file
        self.result_folder = result_folder
        
        # Class variables
        ## handle controls
        self.got_handles = False
        self.oa_temp_handle = -1
        self.zone_temp_handle = -1
        self.zone_htg_tstat_handle = -1
        self.zone_clg_tstat_handle = -1
        ## plot settings
        self.plot_update_interval = 1  # time steps
        ## Data storage
        ### Simulation output
        self.y_outdoor = []
        self.y_radiation = []
        self.y_zone = []
        self.y_occ = []
        self.y_htg = []
        self.y_clg = []
        self.x = []
        self.energy = []
        ### Simulation timesteps (int)
        self.count = 0
        ### Aggregated simulation and control output
        self.variables_df = pd.DataFrame([])
        self.controls_df = pd.DataFrame([])
        ### Time variables
        self.years = []
        self.months = []
        self.days = []
        self.hours = []
        self.minutes = []
        self.current_times = []
        self.actual_date_times = []
        self.actual_times = []
        
    def callback_function(self, state_argument):
        
        if not self.got_handles:
            if not api.exchange.api_data_fully_ready(state_argument):
                return

            # Variable handles
            ## Zone 1
            zone_1 = "Perimeter_ZN_1"
            self.oa_temp_handle = api.exchange.get_variable_handle(
                state_argument,u"SITE OUTDOOR AIR DRYBULB TEMPERATURE", u"ENVIRONMENT")
            self.oa_radiation_handle = api.exchange.get_variable_handle(
                state_argument,u"Site Direct Solar Radiation Rate per Area", u"ENVIRONMENT")
            self.zone_temp_handle = api.exchange.get_variable_handle(
                state_argument, "Zone Mean Air Temperature", zone_1)
            self.zone_htg_tstat_handle = api.exchange.get_variable_handle(
                state_argument, "Zone Thermostat Heating Setpoint Temperature", zone_1)
            self.zone_clg_tstat_handle = api.exchange.get_variable_handle(
                state_argument, "Zone Thermostat Cooling Setpoint Temperature", zone_1)
            self.Facility_Total_Purchased_Electricity_Rate_W_handle = api.exchange.get_variable_handle(
                state_argument, "Zone Air System Sensible Cooling Rate",zone_1)
            self.zone_occupancy_handle = api.exchange.get_variable_handle(
                state_argument, "Zone People Occupant Count", zone_1)
            
            # Actuarator handles
            # Zone 1
            self.handle_clgsetp = api.exchange.get_actuator_handle(
                state_argument, "Schedule:Compact", "Schedule Value", "CLGSETP_SCH")
            self.handle_htgsetp = api.exchange.get_actuator_handle(
                state_argument, "Schedule:Compact", "Schedule Value", "HTGSETP_SCH")
            self.handle_occupancy = api.exchange.get_actuator_handle(
                state_argument, "Schedule:Compact", "Schedule Value", "BLDG_OCC_SCH")
            
            # Check whether all handels are read correctly
            if -1 in [self.oa_temp_handle, self.zone_temp_handle, self.zone_htg_tstat_handle,
                      self.zone_clg_tstat_handle, self.handle_clgsetp,  self.handle_htgsetp,
                      self.Facility_Total_Purchased_Electricity_Rate_W_handle, self.handle_occupancy, 
                      self.oa_radiation_handle, self.zone_occupancy_handle]:
                print("***Invalid handles, check spelling and sensor/actuator availability")
                sys.exit(1)
            self.got_handles = True
            
#             # draw initial plot
#             self.init_plot()

        if api.exchange.warmup_flag(state_argument):
            return
        self.count += 1
        
        # get variables via handle and save variables
        ## outdoor air temperature
        oa_temp = api.exchange.get_variable_value(state_argument, self.oa_temp_handle)
        self.y_outdoor.append(oa_temp)
        ## outdoor solar radiation
        oa_radiation = api.exchange.get_variable_value(state_argument, self.oa_radiation_handle)
        self.y_radiation.append(oa_radiation)
        ## zone air temperature
        zone_temp = api.exchange.get_variable_value(state_argument, self.zone_temp_handle)
        self.y_zone.append(zone_temp)
        ## heating setpoint
        zone_htg_tstat = api.exchange.get_variable_value(state_argument, self.zone_htg_tstat_handle)
        self.y_htg.append(zone_htg_tstat)
        ## cooling setpoint
        zone_clg_tstat = api.exchange.get_variable_value(state_argument, self.zone_clg_tstat_handle)
        self.y_clg.append(zone_clg_tstat)
        ## Electricity rate
        Facility_Total_Purchased_Electricity_Rate_W = api.exchange.get_variable_value(
            state_argument, self.Facility_Total_Purchased_Electricity_Rate_W_handle)
        self.energy.append(Facility_Total_Purchased_Electricity_Rate_W)
        # building occupancy
        zone_occupancy = api.exchange.get_variable_value(state_argument, self.zone_occupancy_handle)
        self.y_occ.append(zone_occupancy)

        # Read Time
        year = api.exchange.year(state)
        month = api.exchange.month(state)
        day = api.exchange.day_of_month(state)
        hour = api.exchange.hour(state)
        minute = api.exchange.minutes(state)
        current_time = api.exchange.current_time(state)
        actual_date_time = api.exchange.actual_date_time(state)
        actual_time = api.exchange.actual_time(state)
  
        # save time
        # TODO: now harcode it to 2009, Year is bogus, seems to be reading the weather file year instead...
        year = 2009
        self.years.append(year)
        self.months.append(month)
        self.days.append(day)
        self.hours.append(hour)
        self.minutes.append(minute)

        self.current_times.append(current_time)
        self.actual_date_times.append(actual_date_time)
        self.actual_times.append(actual_time)

        timedelta = datetime.timedelta()
        if hour >= 24.0:
            hour = 23.0
            timedelta += datetime.timedelta(hours=1)
        if minute >= 60.0:
            minute = 59
            timedelta += datetime.timedelta(minutes=1)

        dt = datetime.datetime(year=year, month=month, day=day, hour=hour, minute=minute)
        dt += timedelta
        self.x.append(dt)        

        
        # get control signals calculated from control components to control the EnergyPlus models 
        ## 1.1 read control signal
        
        while True:
            try:
                clg_spt = pd.read_csv(
                    working_path + f'/dynamic_simulation_result/control_{self.count-1:03}.csv')['clg_spt'][0]
                htg_spt = pd.read_csv(
                    working_path + f'/dynamic_simulation_result/control_{self.count-1:03}.csv')['htg_spt'][0]
                break
            except:
                time.sleep(1)
                
        ## 1.2 set controls
        api.exchange.set_actuator_value(state_argument, self.handle_clgsetp, clg_spt)
        api.exchange.set_actuator_value(state_argument, self.handle_htgsetp, htg_spt)

        # aggregate and varialbes to SMMM
        current_df = pd.DataFrame([[oa_temp, oa_radiation, zone_temp, htg_spt, clg_spt, zone_occupancy, Facility_Total_Purchased_Electricity_Rate_W]], columns = ['oa_temp', 'oa_radiation', 'zone_temp', 'zone_htg_tstat', 'zone_clg_tstat', 'zone_occ','clg_rate'])
        current_df = current_df.T
        current_df.to_csv(f'dynamic_simulation_result/{self.building_id}_{self.count:03}.csv')

### Building Components and Non-Builidng Components Functions
func1(), func2(): building components
func3(): TOU (static data), func4(): advanced model-based control (deep learning algorithm), func5(): Large Language Model interaction (API)

In [4]:
# Building 1
def func1():
    global api, state
    api = EnergyPlusAPI()
    state = api.state_manager.new_state()
    
    building_id = 'bldg_1'
    building_info = None #HOUSEID_1_info
    filename_to_run = "EnergyPlus_models/RefBldgSmallOfficeNew2004_v1.4_7.2_2B_USA_AZ_PHOENIX/RefBldgSmallOfficeNew2004_v1.4_7.2_2B_USA_AZ_PHOENIX_IML_DPC.idf"
    weather_file = 'epw_weather_files/USA_AZ_Tucson.Intl.AP.722740_TMY3.epw'
    result_folder = 'results/RefBldgSmallOfficeNew2004_v1.4_7.2_2B_USA_AZ_PHOENIX_IML_DPC/'

    g = function_n_data(building_id=building_id, building_info=building_info, filename_to_run=filename_to_run, weather_file=weather_file, result_folder=result_folder)

    api.runtime.callback_begin_zone_timestep_after_init_heat_balance(state, g.callback_function)
    api.runtime.run_energyplus(state,
        [
            '-w', g.weather_file,
            '-d', g.result_folder,
            g.filename_to_run
        ]
    )

    # reset the state first to call run_energyplus again
    api.state_manager.reset_state(state)

def func2():
    # import two models
    with open('model/model_fx_IML.pkl', 'rb') as f:
        model_fx = pickle.load(f)

    with open('model/model_fy_IML.pkl', 'rb') as g:
        model_fy = pickle.load(g)

    TOU_IML = pd.read_csv('data/TOU_IML.csv')['power_limit']

    # random perturbation
    while True: 
        number_bldg_1 = 0
        
        try:
            number_bldg_1 = max([int(x[-7:-4]) for x in os.listdir(working_path + '/dynamic_simulation_result') if 'bldg_1' in x])
            
            if number_bldg_1 != 0:
#                 bldg_1 = pd.read_csv(working_path + f'/dynamic_simulation_result/bldg_1_{number_bldg_1:03}.csv')
                
                if number_bldg_1 >= 1:
                    
                    print(f'-----------------{number_bldg_1}----------------')
                    
                    file_list = os.listdir('dynamic_simulation_result/')
                    bldg_file_list = [x for x in file_list if 'bldg' in x][-2:]
                    summary_list = []
                    for file in bldg_file_list:
                        temp_df = pd.read_csv(f'dynamic_simulation_result/{file}')
                        temp_list = list(temp_df.T.iloc[1,:])
                        summary_list.append(temp_list)
                    summary_df = pd.DataFrame(summary_list, columns = temp_df.T.iloc[0,:]).drop(columns = ['zone_htg_tstat'])
                    summary_df_t_minus_1 = summary_df.shift(1).drop(columns = ['clg_rate'])
                    summary_df_t_minus_1.columns = [x + "_t_minus_1" for x in summary_df_t_minus_1.columns]
                    summary_df_total = pd.concat([summary_df ,summary_df_t_minus_1], axis = 1).iloc[1:].reset_index(drop=True)

                    input_list_fx = ['oa_temp', 'oa_radiation',
                                  'zone_temp', 'zone_clg_tstat',
                                  'zone_occ']

                    input_list_fy = ['oa_temp', 'oa_radiation', 'zone_temp', 'zone_clg_tstat', 'zone_occ']

                    def cost_func(u1, u2):
                        fx_predict = summary_df_total[input_list_fx].iloc[-1:]
                        fx_predict['zone_clg_tstat'] = u1

                        x1 = model_fx.predict(fx_predict)[0]

                        fy_predict = summary_df_total[input_list_fy].iloc[-1:]
                        fy_predict['zone_clg_tstat'] = u1
                        fy_predict['zone_temp'] = x1

                        fy_u1_x1_d1 = model_fy.predict(fy_predict)[0]

                        ############################################
                        fx_predict = summary_df_total[input_list_fx].iloc[-1:]
                        fx_predict['zone_clg_tstat'] = u2 
                        fx_predict['zone_temp'] = x1

                        x2 = model_fx.predict(fx_predict)[0]

                        fy_predict = summary_df_total[input_list_fy].iloc[-1:]
                        fy_predict['zone_clg_tstat'] = u2
                        fy_predict['zone_temp'] = x2

                        fy_u2_x2_d2 = model_fy.predict(fy_predict)[0]
                        print(f'x1:{x1},x2:{x2},e1: {fy_u1_x1_d1},e2:{fy_u2_x2_d2}')
#                         print(f'penalty1:{50 * abs(x1-26)**3}, penalty2:{50 * abs(x2-26)**3}')
                        print(f'DR:{TOU_IML[number_bldg_1+1]}, DR:{TOU_IML[number_bldg_1+2]}')
                        
                        if fy_u1_x1_d1 < TOU_IML[number_bldg_1+1] and fy_u2_x2_d2 < TOU_IML[number_bldg_1+2]:
#                             cost_function = fy_u1_x1_d1*TOU_IML[number_bldg_1+1] + fy_u2_x2_d2*TOU_IML[number_bldg_1+2] #+ 5 * abs(x1-24.5)**3 + 5 * abs(x2-24.5)**3
                            cost_function = fy_u1_x1_d1 + fy_u2_x2_d2
                        elif fy_u1_x1_d1 < TOU_IML[number_bldg_1+1] and fy_u2_x2_d2 > TOU_IML[number_bldg_1+2]:
                            cost_function = fy_u1_x1_d1 + TOU_IML[number_bldg_1+2] + (fy_u2_x2_d2 - TOU_IML[number_bldg_1+2]) ** 2
                        elif fy_u1_x1_d1 > TOU_IML[number_bldg_1+1] and fy_u2_x2_d2 < TOU_IML[number_bldg_1+2]:
                            cost_function = fy_u2_x2_d2 + TOU_IML[number_bldg_1+1] + (fy_u1_x1_d1 - TOU_IML[number_bldg_1+1]) ** 2
                        else:
                            cost_function = TOU_IML[number_bldg_1+1] + TOU_IML[number_bldg_1+2] + (fy_u1_x1_d1 - TOU_IML[number_bldg_1+1]) ** 2 + (fy_u2_x2_d2 - TOU_IML[number_bldg_1+2]) ** 2
                            
                        print(f'cost_function:{cost_function}')
                        return (cost_function, x1, x2, fy_u1_x1_d1, fy_u2_x2_d2)

                    u1_list = list(range(22,26 + 1,1))
                    u2_list = list(range(22,26 + 1,1))

                    cost_function_min = 10e8
                    
                    u1_opt = 0
                    u2_opt = 0

                    for u1 in u1_list:
                        for u2 in u2_list:
                            print(f'u1:{u1},u2:{u2}')
                            function_value = cost_func(u1,u2)
                            cost_function_temp = function_value[0]
                            if cost_function_temp < cost_function_min:
                                cost_function_min = cost_function_temp
                                u1_opt = u1
                                u2_opt = u2
                                x1_opt = function_value[1]
                                x2_opt = function_value[2]
                                y1_opt = function_value[3]
                                y2_opt = function_value[4]
                    clg_spt = u1_opt
                    htg_spt = 20
                    print(f'Cooling Setpoint: {clg_spt}')
                    # save optimization results
                    opt_results = pd.DataFrame([x1_opt, x2_opt, u1_opt, u2_opt, y1_opt, y2_opt,cost_function_min]).T
                    opt_results.columns = ['x1','x2','u1','u2','y1','y2','cost_function']
                    opt_results.to_csv(f'data/IML/IML_{number_bldg_1:03}.csv')
#                     print('run_successfully')

            else:
                clg_spt = 26
                htg_spt = 20

            # save the file
            spt = pd.DataFrame([clg_spt,htg_spt], index = ['clg_spt','htg_spt']).T
            spt.to_csv(working_path + f'/dynamic_simulation_result/control_{number_bldg_1:03}.csv')
                
        except:
            spt = pd.DataFrame([26,20], index = ['clg_spt','htg_spt']).T
            spt.to_csv(working_path + f'/dynamic_simulation_result/control_{number_bldg_1:03}.csv')            
            time.sleep(0.5)

In [5]:
# number_bldg_1 = 60

# # import two models
# with open('model/model_fx_IML.pkl', 'rb') as f:
#     model_fx = pickle.load(f)

# with open('model/model_fy_IML.pkl', 'rb') as g:
#     model_fy = pickle.load(g)

# TOU_IML = pd.read_csv('data/TOU_IML.csv')['TOU']

# file_list = os.listdir('dynamic_simulation_result/')
# bldg_file_list = [x for x in file_list if 'bldg' in x][-2:]
# summary_list = []
# for file in bldg_file_list:
#     temp_df = pd.read_csv(f'dynamic_simulation_result/{file}')
#     temp_list = list(temp_df.T.iloc[1,:])
#     summary_list.append(temp_list)
# summary_df = pd.DataFrame(summary_list, columns = temp_df.T.iloc[0,:]).drop(columns = ['zone_htg_tstat'])
# summary_df_t_minus_1 = summary_df.shift(1).drop(columns = ['clg_rate'])
# summary_df_t_minus_1.columns = [x + "_t_minus_1" for x in summary_df_t_minus_1.columns]
# summary_df_total = pd.concat([summary_df ,summary_df_t_minus_1], axis = 1).iloc[1:].reset_index(drop=True)

# input_list_fx = ['oa_temp', 'oa_radiation',
#               'zone_temp', 'zone_clg_tstat',
#               'zone_occ']

# input_list_fy = ['oa_temp', 'oa_radiation', 'zone_temp', 'zone_clg_tstat', 'zone_occ']

# def cost_func(u1, u2):
#     fx_predict = summary_df_total[input_list_fx].iloc[-1:]
#     fx_predict['zone_clg_tstat'] = u1

#     x1 = model_fx.predict(fx_predict)[0]

#     fy_predict = summary_df_total[input_list_fy].iloc[-1:]
#     fy_predict['zone_clg_tstat'] = u1
#     fy_predict['zone_temp'] = x1

#     fy_u1_x1_d1 = model_fy.predict(fy_predict)[0]

#     ############################################
#     fx_predict = summary_df_total[input_list_fx].iloc[-1:]
#     fx_predict['zone_clg_tstat'] = u2 
#     fx_predict['zone_temp'] = x1

#     x2 = model_fx.predict(fx_predict)[0]

#     fy_predict = summary_df_total[input_list_fy].iloc[-1:]
#     fy_predict['zone_clg_tstat'] = u2
#     fy_predict['zone_temp'] = x2

#     fy_u2_x2_d2 = model_fy.predict(fy_predict)[0]
#     print(f'x1:{x1},x2:{x2},e1: {fy_u1_x1_d1*TOU_IML[number_bldg_1+1]},e2:{fy_u2_x2_d2*TOU_IML[number_bldg_1+2]}')
#     print(f'penalty1:{1000 * abs(x1-26)**3}, penalty2:{1000 * abs(x2-26)**3}')
#     cost_function = fy_u1_x1_d1*TOU_IML[number_bldg_1+1] + fy_u2_x2_d2*TOU_IML[number_bldg_1+2] + 75 * (x1-26)**2 + 75 * (x2-26)**2 
#     print(f'cost_function:{cost_function}')
#     return cost_function

# u1_list = list(range(22,30 + 1,1))
# u2_list = list(range(22,30 + 1,1))

# cost_function_min = 10e8
# u1_opt = 0
# u2_opt = 0

# for u1 in u1_list:
#     for u2 in u2_list:
#         print(f'u1:{u1},u2:{u2}')
#         cost_function_temp = cost_func(u1,u2)
#         if cost_function_temp < cost_function_min:
#             cost_function_min = cost_function_temp
#             u1_opt = u1
#             u2_opt = u2
# clg_spt = u1
# htg_spt = 20

In [6]:
# # import two models
# with open('model/model_fx_IML.pkl', 'rb') as f:
#     model_fx = pickle.load(f)

# fx_predict = summary_df_total[input_list_fx].iloc[-1:]
# #     fx_predict['zone_clg_tstat'] = u1

# fx_predict['zone_clg_tstat'] = pd.DataFrame([22])


# x1 = model_fx.predict(fx_predict)[0]

# print(fx_predict)

# print(x1)


In [7]:
# fy_predict = summary_df_total[input_list_fy].iloc[-1:]
# fy_predict['zone_clg_tstat'] = u1
# fy_predict['zone_temp'] = x1

# fy_u1_x1_d1 = model_fy.predict(fy_predict)[0]

# ############################################
# fx_predict = summary_df_total[input_list_fx].iloc[-1:]
# fx_predict['zone_clg_tstat'] = u1
# fx_predict['zone_temp'] = x1

# x2 = model_fx.predict(fx_predict)[0]

# fy_predict = summary_df_total[input_list_fy].iloc[-1:]
# fy_predict['zone_clg_tstat'] = u2
# fy_predict['zone_temp'] = x2

# fy_u2_x2_d2 = model_fy.predict(fy_predict)[0]
# print(f'x1:{x1},x2:{x2},e1: {fy_u1_x1_d1*TOU_IML[number_bldg_1+1]},e2:{fy_u2_x2_d2*TOU_IML[number_bldg_1+2]}')
# print(f'penalty1:{1000 * abs(x1-26)**3}, penalty2:{1000 * abs(x2-26)**3}')
# cost_function = fy_u1_x1_d1*TOU_IML[number_bldg_1+1] + fy_u2_x2_d2*TOU_IML[number_bldg_1+2] + 1000 * (x1-26)**2 + 1000 * (x2-26)**2 
# print(f'cost_function:{cost_function}')

### Run Co-Simulations

In [8]:
# delete all generated dynamic temp files
files = glob.glob(working_path + '/dynamic_simulation_result/*')
for f in files:
    os.remove(f)

if __name__ == '__main__':
    Thread(target = func1).start()
    Thread(target = func2).start()

-----------------1----------------
u1:22,u2:22
-----------------1----------------
u1:22,u2:22
-----------------2----------------
u1:22,u2:22
x1:22.586965649573518,x2:22.50189764342032,e1: 3971.9247206693626,e2:4071.089495181326
DR:5000.0, DR:5000.0
cost_function:8043.014215850689
u1:22,u2:23
x1:22.586965649573518,x2:23.03827888423364,e1: 3971.9247206693626,e2:3405.97908916785
DR:5000.0, DR:5000.0
cost_function:7377.903809837213
u1:22,u2:24
x1:22.586965649573518,x2:24.015214917158975,e1: 3971.9247206693626,e2:1737.7016166282388
DR:5000.0, DR:5000.0
cost_function:5709.626337297601
u1:22,u2:25
x1:22.586965649573518,x2:24.945323203690982,e1: 3971.9247206693626,e2:1187.520770829225
DR:5000.0, DR:5000.0
cost_function:5159.445491498587
u1:22,u2:26
x1:22.586965649573518,x2:25.269379398039085,e1: 3971.9247206693626,e2:303.7061809185866
DR:5000.0, DR:5000.0
cost_function:4275.630901587949
u1:23,u2:22
x1:23.159302592040863,x2:22.543365868888383,e1: 3607.689278954326,e2:3971.9247206693626
DR:5000.

x1:23.159302592040863,x2:22.543365868888383,e1: 3607.689278954326,e2:3971.9247206693626
DR:5000.0, DR:5000.0
cost_function:7579.613999623689
u1:23,u2:23
x1:23.159302592040863,x2:23.075696663074567,e1: 3607.689278954326,e2:3482.1534101300053
DR:5000.0, DR:5000.0
cost_function:7089.842689084331
u1:23,u2:24
x1:23.159302592040863,x2:24.0176136489216,e1: 3607.689278954326,e2:1737.7016166282388
DR:5000.0, DR:5000.0
cost_function:5345.390895582565
u1:23,u2:25
x1:23.159302592040863,x2:24.946938880054276,e1: 3607.689278954326,e2:1187.520770829225
DR:5000.0, DR:5000.0
cost_function:4795.210049783551
u1:23,u2:26
x1:23.159302592040863,x2:25.755355567399715,e1: 3607.689278954326,e2:529.8246554867765
DR:5000.0, DR:5000.0
cost_function:4137.513934441102
u1:24,u2:22
x1:24.041486589643824,x2:22.55976871428483,e1: 1737.7016166282388,e2:3971.9247206693626
DR:5000.0, DR:5000.0
cost_function:5709.626337297601
u1:24,u2:23
x1:24.041486589643824,x2:23.12845725402677,e1: 1737.7016166282388,e2:3570.993595216085

x1:24.03828839222895,x2:25.937485422466462,e1: 1737.7016166282388,e2:444.0964646778294
DR:5000.0, DR:5000.0
cost_function:2181.798081306068
u1:25,u2:22
x1:25.019663872058544,x2:22.57022816920621,e1: 1400.9148884201163,e2:3971.9247206693626
DR:5000.0, DR:5000.0
cost_function:5372.8396090894785
u1:25,u2:23
x1:25.019663872058544,x2:23.158861015977685,e1: 1400.9148884201163,e2:3607.689278954326
DR:5000.0, DR:5000.0
cost_function:5008.604167374442
u1:25,u2:24
x1:25.019663872058544,x2:24.03566924443951,e1: 1400.9148884201163,e2:1737.7016166282388
DR:5000.0, DR:5000.0
cost_function:3138.616505048355
u1:25,u2:25
x1:25.019663872058544,x2:24.99824431302579,e1: 1400.9148884201163,e2:1400.8797175278978
DR:5000.0, DR:5000.0
cost_function:2801.794605948014
u1:25,u2:26
x1:25.019663872058544,x2:25.98642190827988,e1: 1400.9148884201163,e2:514.4079163153572
DR:5000.0, DR:5000.0
cost_function:1915.3228047354735
u1:26,u2:22
x1:26.004306388236483,x2:22.586965649573518,e1: 503.920723575126,e2:3971.924720669

x1:26.004306388236483,x2:25.019663872058544,e1: 503.920723575126,e2:1400.9148884201163
DR:5000.0, DR:5000.0
cost_function:1904.8356119952423
u1:26,u2:26
x1:26.004306388236483,x2:26.004306388236483,e1: 503.920723575126,e2:503.920723575126
DR:5000.0, DR:5000.0
cost_function:1007.841447150252
Cooling Setpoint: 26
-----------------4----------------
u1:22,u2:22
x1:22.586965649573518,x2:22.50189764342032,e1: 3971.9247206693626,e2:4071.089495181326
DR:5000.0, DR:5000.0
cost_function:8043.014215850689
u1:22,u2:23
x1:22.586965649573518,x2:23.03827888423364,e1: 3971.9247206693626,e2:3405.97908916785
DR:5000.0, DR:5000.0
cost_function:7377.903809837213
u1:22,u2:24
x1:22.586965649573518,x2:24.015214917158975,e1: 3971.9247206693626,e2:1737.7016166282388
DR:5000.0, DR:5000.0
cost_function:5709.626337297601
u1:22,u2:25
x1:22.586965649573518,x2:24.945323203690982,e1: 3971.9247206693626,e2:1187.520770829225
DR:5000.0, DR:5000.0
cost_function:5159.445491498587
u1:22,u2:26
x1:22.586965649573518,x2:25.269

x1:22.585497560744546,x2:25.365014606690288,e1: 4087.9957653841457,e2:570.2550032868774
DR:5000.0, DR:5000.0
cost_function:4658.250768671023
u1:23,u2:22
x1:23.159523866078935,x2:22.542610780445326,e1: 3731.825197469882,e2:4087.9957653841457
DR:5000.0, DR:5000.0
cost_function:7819.820962854028
u1:23,u2:23
x1:23.159523866078935,x2:23.0762770137942,e1: 3731.825197469882,e2:3571.211942131465
DR:5000.0, DR:5000.0
cost_function:7303.037139601347
u1:23,u2:24
x1:23.159523866078935,x2:24.02249227087865,e1: 3731.825197469882,e2:1727.0444697380758
DR:5000.0, DR:5000.0
cost_function:5458.869667207958
u1:23,u2:25
x1:23.159523866078935,x2:24.94765819872919,e1: 3731.825197469882,e2:1267.3451073966019
DR:5000.0, DR:5000.0
cost_function:4999.170304866484
u1:23,u2:26
x1:23.159523866078935,x2:25.798926993989106,e1: 3731.825197469882,e2:632.8606292714494
DR:5000.0, DR:5000.0
cost_function:4364.685826741332
u1:24,u2:22
x1:24.041630220608045,x2:22.561920462756902,e1: 1727.0444697380758,e2:4087.9957653841457

...

x1:22.71953470312621,x2:22.642733310269133,e1: 4764.056261208034,e2:4727.933288433897
DR:5000.0, DR:5000.0
cost_function:9491.98954964193
u1:22,u2:23
x1:22.71953470312621,x2:23.044145504958202,e1: 4764.056261208034,e2:4212.902308048843
DR:5000.0, DR:5000.0
cost_function:8976.958569256876
u1:22,u2:24
x1:22.71953470312621,x2:24.035155832824284,e1: 4764.056261208034,e2:3218.3021793406374
DR:5000.0, DR:5000.0
cost_function:7982.358440548671
u1:22,u2:25
x1:22.71953470312621,x2:24.94739332053312,e1: 4764.056261208034,e2:2408.56668713515
DR:5000.0, DR:5000.0
cost_function:7172.622948343183
u1:22,u2:26
x1:22.71953470312621,x2:25.791168086967513,e1: 4764.056261208034,e2:1341.006196891262
DR:5000.0, DR:5000.0
cost_function:6105.062458099296
u1:23,u2:22
x1:23.17804187450851,x2:22.693502666422987,e1: 4557.025289506619,e2:4760.7546679152165
DR:5000.0, DR:5000.0
cost_function:9317.779957421835
u1:23,u2:23
x1:23.17804187450851,x2:23.08679389970584,e1: 4557.025289506619,e2:4421.58074599074
DR:5000.0, 

x1:24.052246194534725,x2:23.13847442773942,e1: 3171.4264358707787,e2:4464.580670357652
DR:5000.0, DR:1369.38151
cost_function:9584798.650224581
u1:24,u2:24
x1:24.052246194534725,x2:24.045423753312924,e1: 3171.4264358707787,e2:3171.4264358707787
DR:5000.0, DR:1369.38151
cost_function:3251906.7228024914
u1:24,u2:25
x1:24.052246194534725,x2:24.965160788191756,e1: 3171.4264358707787,e2:2309.567445530855
DR:5000.0, DR:1369.38151
cost_function:888490.4013158998
u1:24,u2:26
x1:24.052246194534725,x2:25.969557274442227,e1: 3171.4264358707787,e2:1741.0571658812894
DR:5000.0, DR:1369.38151
cost_function:142683.60112065746
u1:25,u2:22
x1:25.00997687246454,x2:22.654764647363546,e1: 2372.51061034888,e2:4742.040536734872
DR:5000.0, DR:1369.38151
cost_function:11378570.802736562
u1:25,u2:23
x1:25.00997687246454,x2:23.161180290986472,e1: 2372.51061034888,e2:4462.687895247995
DR:5000.0, DR:1369.38151
cost_function:9572286.285136363
u1:25,u2:24
x1:25.00997687246454,x2:24.050301193507316,e1: 2372.51061034

x1:25.00115860206238,x2:24.981936968435893,e1: 2399.0924924170586,e2:2353.9613361987695
DR:1369.38151, DR:5000.0
cost_function:1064028.0501565028
u1:25,u2:26
x1:25.00115860206238,x2:26.01874931239528,e1: 2399.0924924170586,e2:1768.3277954314517
DR:1369.38151, DR:5000.0
cost_function:1063442.4166157355
u1:26,u2:22
x1:26.02292326652202,x2:22.662395990286374,e1: 1752.8763436891281,e2:4753.3843036781645
DR:1369.38151, DR:5000.0
cost_function:153191.05327993026
u1:26,u2:23
x1:26.02292326652202,x2:23.162571755995952,e1: 1752.8763436891281,e2:4440.121595405994
DR:1369.38151, DR:5000.0
cost_function:152877.79057165812
u1:26,u2:24
x1:26.02292326652202,x2:24.056581673091276,e1: 1752.8763436891281,e2:3119.9429437249682
DR:1369.38151, DR:5000.0
cost_function:151557.61191997706
u1:26,u2:25
x1:26.02292326652202,x2:25.003992451196204,e1: 1752.8763436891281,e2:2399.0924924170586
DR:1369.38151, DR:5000.0
cost_function:150836.76146866917
u1:26,u2:26
x1:26.02292326652202,x2:26.02292326652202,e1: 1752.876

x1:25.693009313556395,x2:24.990610410730152,e1: 1854.9889665112494,e2:2594.19028560665
DR:5000.0, DR:1407.373164
cost_function:1411797.2422692047
u1:26,u2:26
x1:25.693009313556395,x2:26.020580411796853,e1: 1854.9889665112494,e2:1977.4489928886335
DR:5000.0, DR:1407.373164
cost_function:328248.8128135737
Cooling Setpoint: 22
-----------------495----------------
u1:22,u2:22
x1:22.571495654272088,x2:22.568191251248827,e1: 4671.772491710289,e2:4671.772491710289
DR:5000.0, DR:1407.373164
cost_function:10662382.116411094
u1:22,u2:23
x1:22.571495654272088,x2:23.040381828003902,e1: 4671.772491710289,e2:4169.4141648828845
DR:5000.0, DR:1407.373164
cost_function:7634949.636213836
u1:22,u2:24
x1:22.571495654272088,x2:24.023326607816056,e1: 4671.772491710289,e2:3458.583481339792
DR:5000.0, DR:1407.373164
cost_function:4213542.91161692
u1:22,u2:25
x1:22.571495654272088,x2:24.941503440739346,e1: 4671.772491710289,e2:2483.755889289072
DR:5000.0, DR:1407.373164
cost_function:1164678.9169564398
u1:22,u

x1:23.15713156747823,x2:24.938114645687378,e1: 3863.848727713747,e2:1826.4489719356134
DR:1407.373164, DR:5000.0
cost_function:6037506.017258706
u1:23,u2:26
x1:23.15713156747823,x2:25.894276079704674,e1: 3863.848727713747,e2:1331.8727333191696
DR:1407.373164, DR:5000.0
cost_function:6037011.441020089
u1:24,u2:22
x1:24.042885835708955,x2:22.65216992306494,e1: 2650.4500538458565,e2:4525.744203491715
DR:1407.373164, DR:5000.0
cost_function:1551173.2714363392
u1:24,u2:23
x1:24.042885835708955,x2:23.139614833960547,e1: 2650.4500538458565,e2:3856.4729916114775
DR:1407.373164, DR:5000.0
cost_function:1550504.000224459
u1:24,u2:24
x1:24.042885835708955,x2:24.033694282725996,e1: 2650.4500538458565,e2:2631.370753667799
DR:1407.373164, DR:5000.0
cost_function:1549278.8979865152
u1:24,u2:25
x1:24.042885835708955,x2:24.958401812779744,e1: 2650.4500538458565,e2:1860.0442121208828
DR:1407.373164, DR:5000.0
cost_function:1548507.5714449682
u1:24,u2:26
x1:24.042885835708955,x2:25.970963110904872,e1: 26